In [17]:
"""
DCF Valuation Tool
------------------
This script performs a Discounted Cash Flow (DCF) valuation using historical
Free Cash Flow (FCF) data from yfinance.

Features:
- Computes historical CAGR of FCF
- Projects future cash flows
- Discounts them using WACC
- Computes terminal value
- Outputs fair value: BUY OR SELL

NOTE: This is a simplified financial model for educational purposes.
"""


import pandas as pd
import yfinance as yf
import numpy as np

In [18]:
ticker="AAPL" #change this
projection_years = 10
wacc=0.11
g = 0.02 #Perpetual Growth rate

stock = yf.Ticker(ticker)
cf = stock.cashflow
fcf = cf.loc["Free Cash Flow"]
fcf=fcf.dropna()

if len(fcf) < 2:
    raise ValueError("Not enough FCF data to compute CAGR. At least 2 years are required.")


# Hystorical CAGR
fcf_initial = fcf.iloc[-1] # old
fcf_final = fcf.iloc[0] # recent
years = len(fcf) - 1 
cagr = (fcf_final / fcf_initial) ** (1 / years) - 1 
print(f"\n Hystorical CAGR Free Cash Flow: {cagr:.1%}")
    


 Hystorical CAGR Free Cash Flow: -3.9%


In [19]:
# DCF USING HISTORICAL CAGR

future_fcf = []
for year in range(1, projection_years + 1):
    projected = fcf_final * ((1 + cagr) ** year)
    future_fcf.append(projected)


discounted_fcf = []
for year, fcf_value in enumerate(future_fcf, start=1):
    pv = fcf_value / (1 + wacc) ** year
    discounted_fcf.append(pv)



last_projected_fcf = future_fcf[-1] 
# Terminal Value 
terminal_value = (last_projected_fcf * (1 + g)) / (wacc - g)
#Discounted terminal value
pv_terminal_value = terminal_value / ((1 + wacc) ** projection_years)

#BALANCE SHEET
bs = stock.balance_sheet 
def safe_get(bs, key):
    return bs.loc[key].iloc[0] if key in bs.index else 0

cash = safe_get(bs, "Cash And Cash Equivalents")
short_debt = safe_get(bs, "Current Debt")
long_debt = safe_get(bs, "Long Term Debt")

net_cash = cash - (short_debt + long_debt)

fair_value = sum(discounted_fcf) + pv_terminal_value + net_cash

shares = stock.info.get("sharesOutstanding", None)
fair_value_per_share = fair_value / shares

In [20]:

current_price = stock.info.get("currentPrice", None)
if not shares or not current_price:
    raise ValueError("Missing market data.")
    
difference = (fair_value_per_share - current_price) / current_price * 100
# BUY / SELL / HOLD
if fair_value_per_share > current_price * 1.03:
     verdict = "BUY"
elif fair_value_per_share < current_price * 0.97: 
    verdict = "SELL"
else: 
    verdict = "HOLD"
# Output 
print("\n" + "="*50) 
print(f"DCF VALUATION SUMMARY FOR {ticker.upper()}")
print("="*50)
print(f"Fair Value per Share: {fair_value_per_share:,.2f}") 
print(f"Current Market Price: {current_price:,.2f}")
print(f"Valuation Difference: {difference:+.2f}%")
print(f"Recommendation: {verdict}") 
print("="*50)


DCF VALUATION SUMMARY FOR AAPL
Fair Value per Share: 46.74
Current Market Price: 260.48
Valuation Difference: -82.06%
Recommendation: SELL


In [21]:

# ============================
# USER-DEFINED GROWTH MODEL
# ============================

growth_input = float(input("Input annual growth rate FCF (ex. 0.05 for 5%): "))

stock = yf.Ticker(ticker)


cf = stock.cashflow
fcf = cf.loc["Free Cash Flow"].dropna()

if len(fcf) < 1:
    raise ValueError("No enough Free Cash Flow.")

fcf_last = fcf.iloc[0]  



future_fcf = []

for year in range(1, projection_years + 1):
    projected = fcf_last * ((1 + growth_input) ** year)
    future_fcf.append(projected)



discounted_fcf = []

for year, fcf_value in enumerate(future_fcf, start=1):
    pv = fcf_value / (1 + wacc) ** year
    discounted_fcf.append(pv)


# TERMINAL VALUE

last_projected_fcf = future_fcf[-1]

terminal_value = (last_projected_fcf * (1 + g)) / (wacc - g)
pv_terminal_value = terminal_value / ((1 + wacc) ** projection_years)


#  CASH & DEBT

bs = stock.balance_sheet

def safe_get(bs, key):
    return bs.loc[key].iloc[0] if key in bs.index else 0

cash = safe_get(bs, "Cash And Cash Equivalents")
short_debt = safe_get(bs, "Current Debt")
long_debt = safe_get(bs, "Long Term Debt")

net_cash = cash - (short_debt + long_debt)


# 6. FAIR VALUE

fair_value = sum(discounted_fcf) + pv_terminal_value + net_cash

shares = stock.info["sharesOutstanding"]
fair_value_per_share = fair_value / shares


current_price = stock.info["currentPrice"]
difference = (fair_value_per_share - current_price) / current_price * 100

if fair_value_per_share > current_price * 1.03:
    verdict = "BUY"
elif fair_value_per_share < current_price * 0.97:
    verdict = "SELL"
else:
    verdict = "HOLD"

# ============================
# 8. OUTPUT VISUALE
# ============================
print("\n" + "="*55)
print(f"DCF VALUATION (CUSTOM GROWTH) FOR {ticker}")
print("="*55)
print(f"Annual growth rate used:   {growth_input*100:.2f}%")
print(f"WACC:                          {wacc*100:.2f}%")
print(f"Fair Value per Share:          {fair_value_per_share:,.2f}")
print(f"Current Market Price:          {current_price:,.2f}")
print(f"Valuation Difference:          {difference:+.2f}%")
print(f"Recommendation:                {verdict}")
print("="*55)


Input annual growth rate FCF (ex. 0.05 for 5%):  0.15



DCF VALUATION (CUSTOM GROWTH) FOR AAPL
Annual growth rate used:   15.00%
WACC:                          11.00%
Fair Value per Share:          186.52
Current Market Price:          260.48
Valuation Difference:          -28.39%
Recommendation:                SELL
